# Iceberg Merge Pipeline

Инкрементальный MERGE из staging в основную таблицу с compaction и управлением снапшотами.

## Режимы запуска:
- **Local** (по умолчанию) — быстро, ~3 сек
- **YARN** — для продакшена: замените `%run` на `%run /home/jovyan/work/init_spark.py yarn`

In [ ]:
%run /home/jovyan/work/init_spark.py

### Шаг 0: Создание баз данных

In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS analytics")
spark.sql("CREATE DATABASE IF NOT EXISTS staging")
print("Databases created.")

### Конфигурация

In [ ]:
TARGET = "analytics.customer_data"
SOURCE = "staging.daily_updates"
DATE = "2025-09-01"

### Шаг 1: Инкрементальный MERGE (Upsert + Soft Delete)

In [ ]:
sql = "MERGE INTO " + TARGET + " AS target " +
    "USING " + SOURCE + " AS source " +
    "ON target.id = source.id " +
    "WHEN MATCHED AND source.is_deleted = true THEN DELETE " +
    "WHEN MATCHED THEN UPDATE SET " +
    "target.name = source.name, target.email = source.email, target.last_updated = current_timestamp() " +
    "WHEN NOT MATCHED AND source.is_deleted = false THEN INSERT (id, name, email, last_updated) " +
    "VALUES (source.id, source.name, source.email, current_timestamp())"

spark.sql(sql)

count = spark.sql("SELECT COUNT(*) FROM " + TARGET).collect()[0][0]
print(f"Rows in {TARGET} after merge: {count}")

### Шаг 2: Compaction — уплотнение data-файлов

In [ ]:
MIN_INPUT_FILES = "5"
TARGET_FILE_SIZE_BYTES = "268435456"

sql = "CALL iceberg_catalog.system.rewrite_data_files(" +
    "table => '" + TARGET + "', " +
    "options => map('min-input-files', '" + MIN_INPUT_FILES + "', " +
    "'target-file-size-bytes', '" + TARGET_FILE_SIZE_BYTES + "'))"

spark.sql(sql)
print(f"Compaction completed for {TARGET}")

### Шаг 3: Expire старых снапшотов

In [ ]:
RETENTION_DAYS = "7"

sql = "CALL iceberg_catalog.system.expire_snapshots(" +
    "table => '" + TARGET + "', " +
    "older_than => timestamp_add('day', -" + RETENTION_DAYS + ", current_timestamp()), " +
    "retain_last => 1)"

spark.sql(sql)
print(f"Snapshots older than {RETENTION_DAYS} days expired for {TARGET}")

### Верификация

In [ ]:
spark.sql("SELECT * FROM " + TARGET + "$history").show(truncate=False)
spark.sql("SELECT * FROM " + TARGET + "$metadata").show(truncate=False)

In [ ]:
spark.stop()